# PCB Defect Detection YOLOv8s (Google Colab)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 29.4 MB/s eta 0:00:00


In [ ]:
import os, glob, shutil
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
ZIP_PATH='/content/drive/MyDrive/PCB_Datasets/archive(1).zip'
EXTRACT_DIR='/content/pcb_dataset'
!rm -rf /content/pcb_dataset
!mkdir -p "$EXTRACT_DIR"
!unzip -q "$ZIP_PATH" -d "$EXTRACT_DIR"
print('Dataset selesai diekstrak')

Dataset selesai diekstrak


In [ ]:
!find /content/pcb_dataset -maxdepth 3 -type d

/content/pcb_dataset
/content/pcb_dataset/pcb-defect-dataset
/content/pcb_dataset/pcb-defect-dataset/test
/content/pcb_dataset/pcb-defect-dataset/test/images
/content/pcb_dataset/pcb-defect-dataset/test/labels
/content/pcb_dataset/pcb-defect-dataset/val
/content/pcb_dataset/pcb-defect-dataset/val/images
/content/pcb_dataset/pcb-defect-dataset/val/labels
/content/pcb_dataset/pcb-defect-dataset/train
/content/pcb_dataset/pcb-defect-dataset/train/images
/content/pcb_dataset/pcb-defect-dataset/train/labels


In [ ]:
BASE='/content/pcb_dataset/pcb-defect-dataset'
print('Train images:', len(glob.glob(f'{BASE}/train/images/*.jpg')))
print('Train labels:', len(glob.glob(f'{BASE}/train/labels/*.txt')))
print('Val images:', len(glob.glob(f'{BASE}/val/images/*.jpg')))
print('Val labels:', len(glob.glob(f'{BASE}/val/labels/*.txt')))
print('Test images:', len(glob.glob(f'{BASE}/test/images/*.jpg')))
print('Test labels:', len(glob.glob(f'{BASE}/test/labels/*.txt')))

Train images: 8534
Train labels: 8534
Val images: 1066
Val labels: 1066
Test images: 1068
Test labels: 1068


In [ ]:
for split in ['train','val','test']:
    img_dir=f'{BASE}/{split}/images'
    lbl_dir=f'{BASE}/{split}/labels'
    fixed=0
    for img_path in glob.glob(f'{img_dir}/*.jpg'):
        img_name=os.path.splitext(os.path.basename(img_path))[0]
        expected=f'{lbl_dir}/{img_name}.txt'
        alt=f'{lbl_dir}/l_{img_name}.txt'
        if (not os.path.exists(expected)) and os.path.exists(alt):
            shutil.copy(alt, expected)
            fixed+=1
    print(split, 'fixed:', fixed)

train fixed: 1706
val fixed: 26
test fixed: 25


In [ ]:
!find /content/pcb_dataset -name '*.cache' -delete

In [ ]:
yaml_text='''
path: /content/pcb_dataset/pcb-defect-dataset
train: train/images
val: val/images
test: test/images

nc: 6

names:
  0: mouse_bite
  1: spur
  2: missing_hole
  3: short
  4: open_circuit
  5: spurious_copper
'''
DATA_YAML=f'{BASE}/data.yaml'
import os
os.makedirs(os.path.dirname(DATA_YAML), exist_ok=True)
open(DATA_YAML,'w').write(yaml_text)
print(open(DATA_YAML).read())


path: /content/pcb_dataset/pcb-defect-dataset
train: train/images
val: val/images
test: test/images

nc: 6

names:
  0: mouse_bite
  1: spur
  2: missing_hole
  3: short
  4: open_circuit
  5: spurious_copper



In [ ]:
model=YOLO('yolov8s.pt')
results=model.train(data=DATA_YAML,epochs=50,imgsz=640,batch=16,workers=4,patience=20,project='/content/PCB_Research',name='YOLOv8s_PCB',exist_ok=True)

Ultralytics 8.4.63 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/pcb_dataset/pcb-defect-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=YOLOv8s_PCB, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap

In [ ]:
best='/content/PCB_Research/YOLOv8s_PCB/weights/best.pt'
model=YOLO(best)
metrics=model.val(data=DATA_YAML,split='test')
print('mAP50:',metrics.box.map50)
print('mAP50-95:',metrics.box.map)

Ultralytics 8.4.63 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 11,127,906 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 109.8±163.2 MB/s, size: 93.9 KB)
val: Scanning /content/pcb_dataset/pcb-defect-dataset/test/labels... 854 images, 214 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1068/1068 548.0it/s 1.9s
val: New cache created: /content/pcb_dataset/pcb-defect-dataset/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 67/67 3.9it/s 17.2s
                   all       1068       1717      0.781      0.986      0.864       0.53
            mouse_bite        135        270      0.794      0.993      0.872      0.529
                  spur        143        290      0.821      0.983      0.906      0.552
          missing_hole        148        292      0.756          1       0.86      0.548
                 short        146 

In [ ]:
model.predict(source=f'{BASE}/test/images',save=True,conf=0.25,project='/content/PCB_Research',name='YOLOv8s_Predictions',exist_ok=True)


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

image 1/1068 /content/pcb_dataset/pcb-defect-dataset/test/images/l_light_01_missing_hole_04_2_600.jpg: 640x640 2 missing_holes, 16.2ms
image 2/1068 /content/pcb_dataset/pcb-defect-dataset/test/images/l_light_01_missing_hole_07_2_600.jpg: 640x640 2 missing_holes, 16.2ms
image 3/1068 /content/pcb_dataset/pcb-defect-dataset/test/images/l_light_01_missing_hole_08_1_600.jpg: 640x640 1 missing_hole, 16.1ms
image 4/1068 /content/pcb_dataset/pcb-defect-dataset/t

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'mouse_bite', 1: 'spur', 2: 'missing_hole', 3: 'short', 4: 'open_circuit', 5: 'spurious_copper'}
 obb: None
 orig_img: array([[[135, 148, 156],
         [132, 145, 153],
         [131, 146, 155],
         ...,
         [127, 148, 140],
         [125, 145, 140],
         [124, 144, 139]],
 
        [[134, 147, 155],
         [131, 144, 152],
         [129, 144, 153],
         ...,
         [136, 156, 151],
         [137, 157, 152],
         [137, 157, 152]],
 
        [[133, 146, 154],
         [131, 144, 152],
         [129, 144, 153],
         ...,
         [131, 150, 147],
         [133, 150, 147],
         [133, 150, 147]],
 
        ...,
 
        [[178, 174, 180],
         [178, 174, 180],
         [179, 175, 181],
         ...,
         [187, 184, 186],
         [187, 184, 186],
         [187, 184, 186]],
 
        [[177, 173, 179

In [ ]:
!zip -r /content/pcb_yolov8s_results.zip /content/PCB_Research/YOLOv8s_PCB /content/PCB_Research/YOLOv8s_Predictions

  adding: content/PCB_Research/YOLOv8s_PCB/ (stored 0%)
  adding: content/PCB_Research/YOLOv8s_PCB/labels.jpg (deflated 26%)
  adding: content/PCB_Research/YOLOv8s_PCB/confusion_matrix.png (deflated 21%)
  adding: content/PCB_Research/YOLOv8s_PCB/val_batch2_labels.jpg (deflated 11%)
  adding: content/PCB_Research/YOLOv8s_PCB/val_batch1_labels.jpg (deflated 10%)
  adding: content/PCB_Research/YOLOv8s_PCB/train_batch21360.jpg (deflated 6%)
  adding: content/PCB_Research/YOLOv8s_PCB/confusion_matrix_normalized.png (deflated 21%)
  adding: content/PCB_Research/YOLOv8s_PCB/train_batch21361.jpg (deflated 8%)
  adding: content/PCB_Research/YOLOv8s_PCB/results.csv (deflated 60%)
  adding: content/PCB_Research/YOLOv8s_PCB/val_batch0_labels.jpg (deflated 11%)
  adding: content/PCB_Research/YOLOv8s_PCB/val_batch0_pred.jpg (deflated 10%)
  adding: content/PCB_Research/YOLOv8s_PCB/BoxP_curve.png (deflated 11%)
  adding: content/PCB_Research/YOLOv8s_PCB/args.yaml (deflated 52%)
  adding: content/PCB